<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Build step 4</div>
<div style="font-size:24px;font-weight:700;">&#128207; Semantic layer and governance</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Three Unity Catalog metric views (one definition of every KPI), primary and foreign keys, column descriptions and governed tags.</p>


In [0]:
# Parameters come from the bundle's build job: the catalog and the three schemas the bundle created.
# This sets USE CATALOG and the session variables bronze_schema, silver_schema and gold_schema.
from bean_and_brew import init, fill

cfg = init(spark, dbutils)


In [0]:
# Metric views: the YAML is the original definition, pointed at the bundle's catalog and schemas
METRIC_VIEWS = {
    'store_sales_metrics': r'''CREATE VIEW IF NOT EXISTS __CATALOG__.__GOLD__.store_sales_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Certified Bean & Brew store performance KPIs: net sales, traffic, average ticket, margin, labor and loyalty mix. Use for any store-level or regional sales question.'
AS $$
version: 1.1
source: __CATALOG__.__GOLD__.store_daily_performance
comment: Certified Bean & Brew store performance KPIs. One source row = one store-day. Excludes voided tickets.
dimensions:
  - name: business_date
    expr: business_date
    display_name: Business Date
    comment: Store-local business date.
    format:
      type: date
      date_format: year_month_day
    synonyms: [date, day, sales date]
  - name: business_month
    expr: DATE_TRUNC('MONTH', business_date)
    display_name: Month
    comment: Calendar month of the business date.
    format:
      type: date
      date_format: locale_short_month
    synonyms: [month, sales month]
  - name: business_week
    expr: DATE_TRUNC('WEEK', business_date)
    display_name: Week
    comment: Week (Monday start) of the business date.
    synonyms: [week]
  - name: day_of_week
    expr: date_format(business_date, 'EEEE')
    display_name: Day of Week
    synonyms: [weekday]
  - name: store_name
    expr: store_name
    display_name: Store
    comment: Customer-facing store name.
    synonyms: [store, location, cafe, shop]
  - name: store_id
    expr: store_id
    display_name: Store ID
  - name: city
    expr: city
    display_name: City
  - name: region
    expr: region
    display_name: Region
    comment: Operating region (Pacific Northwest, California, Southwest, Mountain).
    synonyms: [market, territory]
  - name: store_format
    expr: store_format
    display_name: Store Format
    comment: Flagship Cafe, Cafe, Drive-Thru, or Express Kiosk.
    synonyms: [format, store type]
  - name: is_comp_store
    expr: is_comp_store
    display_name: Comp Store
    comment: True when the store has been open 365+ days (comparable-store sales).
    synonyms: [comparable store, same store]
measures:
  - name: net_sales
    expr: SUM(net_sales)
    display_name: Net Sales
    comment: Gross sales minus discounts, pre-tax. Primary revenue KPI.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [revenue, sales, total sales, turnover]
  - name: transactions
    expr: SUM(transactions)
    display_name: Transactions
    comment: Completed tickets, used as a proxy for customer traffic.
    format:
      type: number
      decimal_places: {type: exact, places: 0}
    synonyms: [tickets, orders, traffic, visits, customer count]
  - name: avg_ticket
    expr: SUM(net_sales) / SUM(transactions)
    display_name: Average Ticket
    comment: Net sales divided by transactions.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 2}
    synonyms: [average order value, AOV, average check, basket size]
  - name: gross_profit
    expr: SUM(gross_profit)
    display_name: Gross Profit
    comment: Net sales minus cost of goods sold.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [margin dollars, profit]
  - name: gross_margin_pct
    expr: SUM(gross_profit) / SUM(net_sales)
    display_name: Gross Margin %
    comment: Gross profit as a share of net sales.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [margin, GM%]
  - name: labor_cost_pct
    expr: SUM(labor_cost) / SUM(net_sales)
    display_name: Labor Cost %
    comment: Labor cost as a share of net sales. Target is below 30%.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [labor percent, labor ratio, wage ratio]
  - name: loyalty_mix_pct
    expr: SUM(loyalty_transactions) / SUM(transactions)
    display_name: Loyalty Mix %
    comment: Share of tickets with a Bean & Brew Rewards member attached.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [rewards penetration, loyalty share]
  - name: digital_mix_pct
    expr: SUM(digital_transactions) / SUM(transactions)
    display_name: Digital Mix %
    comment: Share of tickets ordered via Mobile App or Delivery.
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [mobile share, digital share, app orders share]
  - name: discount_rate
    expr: SUM(discount_amount) / SUM(gross_sales)
    display_name: Discount Rate
    comment: Promotional discounts as a share of gross sales.
    format:
      type: percentage
      decimal_places: {type: exact, places: 2}
    synonyms: [promo rate, markdown rate]
  - name: store_days
    expr: COUNT(1)
    display_name: Store Operating Days
    comment: Number of store-days with sales.
$$''',
    'product_sales_metrics': r'''CREATE VIEW IF NOT EXISTS __CATALOG__.__GOLD__.product_sales_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Certified Bean & Brew menu/product KPIs at line-item grain with star-schema joins to product and store dimensions. Use for product mix, category, daypart, and channel questions.'
AS $$
version: 1.1
source: __CATALOG__.__SILVER__.fact_sales_line_items
comment: Product mix KPIs. Grain = one POS line item. Joins product and store dimensions (many-to-one, PK-enforced).
joins:
  - name: product
    source: __CATALOG__.__SILVER__.dim_product
    on: source.product_id = product.product_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
  - name: store
    source: __CATALOG__.__SILVER__.dim_store
    on: source.store_id = store.store_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
dimensions:
  - name: business_date
    expr: source.business_date
    display_name: Business Date
    synonyms: [date, day]
  - name: business_month
    expr: DATE_TRUNC('MONTH', source.business_date)
    display_name: Month
    synonyms: [month]
  - name: business_hour
    expr: source.business_hour
    display_name: Hour of Day
    comment: Hour (0-23) the ticket was rung up, store-local.
    synonyms: [hour, time of day]
  - name: daypart
    expr: |-
      CASE WHEN source.business_hour < 11 THEN 'Morning Rush (6-11)'
           WHEN source.business_hour < 14 THEN 'Midday (11-14)'
           WHEN source.business_hour < 17 THEN 'Afternoon (14-17)'
           ELSE 'Evening (17+)' END
    display_name: Daypart
    comment: Operational daypart bucket.
    synonyms: [shift, time of day bucket]
  - name: order_channel
    expr: source.order_channel
    display_name: Order Channel
    comment: In-Store, Mobile App, Drive-Thru, or Delivery.
    synonyms: [channel, ordering method]
  - name: product_name
    expr: product.product_name
    display_name: Product
    synonyms: [item, drink, menu item, sku]
  - name: category
    expr: product.category
    display_name: Category
    comment: Menu category (Espresso Drinks, Brewed Coffee, Cold Drinks, Tea, Bakery, Breakfast, Seasonal, Retail Beans).
    synonyms: [product category, menu category]
  - name: serving_type
    expr: product.serving_type
    display_name: Serving Type
    comment: Hot, Iced, Food, or Retail.
    synonyms: [hot or iced, temperature]
  - name: is_beverage
    expr: product.is_beverage
    display_name: Is Beverage
  - name: store_name
    expr: store.store_name
    display_name: Store
    synonyms: [store, location, cafe]
  - name: region
    expr: store.region
    display_name: Region
    synonyms: [market]
  - name: is_loyalty_member
    expr: source.loyalty_member_id IS NOT NULL
    display_name: Loyalty Member Purchase
    synonyms: [rewards member, member purchase]
measures:
  - name: units_sold
    expr: SUM(source.quantity)
    display_name: Units Sold
    comment: Total item quantity sold.
    format:
      type: number
      decimal_places: {type: exact, places: 0}
    synonyms: [units, quantity, items sold, cups sold]
  - name: net_sales
    expr: SUM(source.net_sales)
    display_name: Net Sales
    comment: Gross sales minus discounts, pre-tax.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [revenue, sales]
  - name: gross_profit
    expr: SUM(source.net_sales) - SUM(source.cost_of_goods)
    display_name: Gross Profit
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [profit, margin dollars]
  - name: gross_margin_pct
    expr: (SUM(source.net_sales) - SUM(source.cost_of_goods)) / SUM(source.net_sales)
    display_name: Gross Margin %
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [margin, GM%]
  - name: discount_amount
    expr: SUM(source.discount_amount)
    display_name: Discounts
    comment: Promotional discounts given (LOYALTY10, HAPPYHOUR).
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
    synonyms: [promo cost, markdowns]
  - name: tickets
    expr: COUNT(DISTINCT source.transaction_id)
    display_name: Tickets Containing Item
    comment: Distinct tickets that included the item(s) in the current grouping.
    synonyms: [orders, attach count]
$$''',
    'customer_metrics': r'''CREATE VIEW IF NOT EXISTS __CATALOG__.__GOLD__.customer_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Certified Bean & Brew Rewards loyalty KPIs: member counts, lifetime value, visit frequency, and mobile adoption by tier and segment.'
AS $$
version: 1.1
source: __CATALOG__.__GOLD__.customer_360
comment: Loyalty member KPIs. Grain = one Bean & Brew Rewards member.
dimensions:
  - name: loyalty_tier
    expr: loyalty_tier
    display_name: Loyalty Tier
    comment: Green (entry), Gold, Reserve (top tier).
    synonyms: [tier, rewards tier, membership level]
  - name: customer_segment
    expr: customer_segment
    display_name: Customer Segment
    comment: Champion (top 10% spend), Loyal Regular, Occasional, Light (bottom 20%).
    synonyms: [segment, value segment, RFM segment]
  - name: favorite_product
    expr: favorite_product
    display_name: Favorite Product
    synonyms: [usual order, go-to drink]
  - name: favorite_store_id
    expr: favorite_store_id
    display_name: Favorite Store ID
  - name: signup_year
    expr: year(signup_date)
    display_name: Signup Year
    synonyms: [cohort, enrollment year]
measures:
  - name: member_count
    expr: COUNT(1)
    display_name: Members
    synonyms: [members, customers, number of members]
  - name: total_lifetime_value
    expr: SUM(lifetime_net_sales)
    display_name: Total Lifetime Value
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 0}
      abbreviation: compact
    synonyms: [member revenue, total LTV]
  - name: avg_lifetime_value
    expr: AVG(lifetime_net_sales)
    display_name: Avg Lifetime Value
    comment: Average net sales per member since 2025-01-01.
    format:
      type: currency
      currency_code: USD
      decimal_places: {type: exact, places: 2}
    synonyms: [LTV, CLV, average customer value]
  - name: avg_visits
    expr: AVG(visits)
    display_name: Avg Visits per Member
    format:
      type: number
      decimal_places: {type: exact, places: 1}
    synonyms: [visit frequency, frequency]
  - name: avg_mobile_share
    expr: AVG(mobile_share)
    display_name: Avg Mobile Order Share
    format:
      type: percentage
      decimal_places: {type: exact, places: 1}
    synonyms: [app adoption, mobile adoption]
$$''',
}

for name, ddl in METRIC_VIEWS.items():
    spark.sql(fill(ddl, cfg))
    print(f"Created metric view {cfg['gold_schema']}.{name}")

In [0]:
# Keys, column descriptions and governed tags
CONSTRAINTS = [
 [
  "__SILVER__.dim_store",
  "pk_dim_store",
  "PRIMARY KEY (store_id)"
 ],
 [
  "__SILVER__.dim_product",
  "pk_dim_product",
  "PRIMARY KEY (product_id)"
 ],
 [
  "__SILVER__.dim_loyalty_member",
  "pk_dim_loyalty_member",
  "PRIMARY KEY (member_id)"
 ],
 [
  "__SILVER__.fact_transactions",
  "pk_fact_transactions",
  "PRIMARY KEY (transaction_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "pk_fact_sales_line_items",
  "PRIMARY KEY (line_item_id)"
 ],
 [
  "__GOLD__.customer_360",
  "pk_customer_360",
  "PRIMARY KEY (member_id)"
 ],
 [
  "__SILVER__.dim_loyalty_member",
  "fk_member_home_store",
  "FOREIGN KEY (home_store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_transactions",
  "fk_txn_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_transactions",
  "fk_txn_member",
  "FOREIGN KEY (loyalty_member_id) REFERENCES __SILVER__.dim_loyalty_member(member_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "fk_line_txn",
  "FOREIGN KEY (transaction_id) REFERENCES __SILVER__.fact_transactions(transaction_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "fk_line_product",
  "FOREIGN KEY (product_id) REFERENCES __SILVER__.dim_product(product_id)"
 ],
 [
  "__SILVER__.fact_sales_line_items",
  "fk_line_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_inventory_daily",
  "fk_inv_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ],
 [
  "__SILVER__.fact_labor_daily",
  "fk_labor_store",
  "FOREIGN KEY (store_id) REFERENCES __SILVER__.dim_store(store_id)"
 ]
]

COLUMN_COMMENTS = {
 "__GOLD__.customer_360": {
  "loyalty_tier": "Rewards tier: Green (entry), Gold, Reserve (top tier).",
  "home_store_id": "Store the member enrolled at. FK to dim_store.",
  "lifetime_net_sales": "Total net sales (pre-tax, after discounts) across all non-voided visits since 2025-01-01, USD.",
  "days_since_last_visit": "Days between last visit and 2026-10-19 (data as-of date).",
  "customer_segment": "Value segment by lifetime spend percentile: Champion (top 10%), Loyal Regular (next 30%), Occasional (next 40%), Light (bottom 20%)."
 },
 "__GOLD__.inventory_current_status": {
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "stock_status": "REORDER NOW = below reorder point; LOW = below 1.5x reorder point; OK otherwise."
 },
 "__GOLD__.loyalty_promo_targets": {
  "loyalty_tier": "Rewards tier: Green (entry), Gold, Reserve (top tier).",
  "customer_segment": "Value segment by lifetime spend percentile: Champion (top 10%), Loyal Regular (next 30%), Occasional (next 40%), Light (bottom 20%).",
  "lifetime_net_sales": "Total net sales (pre-tax, after discounts) across all non-voided visits since 2025-01-01, USD.",
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain."
 },
 "__GOLD__.product_sales_daily": {
  "business_date": "Store-local business date of the sale.",
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "category": "Menu category: Espresso Drinks, Brewed Coffee, Cold Drinks, Tea, Bakery, Breakfast, Seasonal, Retail Beans.",
  "is_beverage": "True for drink categories; false for food and retail beans."
 },
 "__GOLD__.store_daily_performance": {
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "store_format": "Store format: Flagship Cafe, Cafe, Drive-Thru, or Express Kiosk.",
  "is_comp_store": "True when the store has been open at least 365 days as of business_date (comparable-store sales).",
  "transactions": "Count of completed (non-voided) tickets. Proxy for customer traffic.",
  "net_sales": "Net sales = gross sales - discounts, pre-tax, USD. Primary revenue measure.",
  "gross_profit": "net_sales - cost_of_goods, USD.",
  "avg_ticket": "Average ticket = net_sales / transactions, USD.",
  "digital_transactions": "Tickets ordered via Mobile App or Delivery.",
  "labor_cost": "worked_hours * avg_hourly_wage, USD (excludes benefits).",
  "labor_cost_pct": "labor_cost / net_sales. Target is below 0.30."
 },
 "__SILVER__.dim_loyalty_member": {
  "email": "Member email address (PII). Lower-cased.",
  "home_store_id": "Store the member enrolled at. FK to dim_store.",
  "loyalty_tier": "Rewards tier: Green (entry), Gold, Reserve (top tier)."
 },
 "__SILVER__.dim_product": {
  "category": "Menu category: Espresso Drinks, Brewed Coffee, Cold Drinks, Tea, Bakery, Breakfast, Seasonal, Retail Beans.",
  "list_price_2025": "Menu list price in USD under the 2025 price book.",
  "list_price_2026": "Menu list price in USD under the 2026 price book (+5%, effective 2026-01-05).",
  "unit_cost": "Standard cost of goods per unit in USD (ingredients + packaging).",
  "standard_margin_pct": "Standard gross margin at 2025 list price: (list_price - unit_cost) / list_price.",
  "is_beverage": "True for drink categories; false for food and retail beans."
 },
 "__SILVER__.dim_store": {
  "store_id": "Unique store identifier (e.g. S001). Primary key.",
  "store_name": "Customer-facing store name.",
  "region": "Operating region: Pacific Northwest, California, Southwest, Mountain.",
  "store_format": "Store format: Flagship Cafe, Cafe, Drive-Thru, or Express Kiosk.",
  "open_date": "Date the store opened to the public. Stores open < 12 months are considered new (non-comp) stores.",
  "seating_capacity": "Number of customer seats."
 },
 "__SILVER__.fact_inventory_daily": {
  "is_below_reorder_point": "True when on-hand quantity is below the reorder point and a replenishment order is required."
 },
 "__SILVER__.fact_labor_daily": {
  "labor_cost": "worked_hours * avg_hourly_wage, USD (excludes benefits)."
 },
 "__SILVER__.fact_sales_line_items": {
  "business_date": "Store-local business date of the sale.",
  "business_hour": "Hour of day (0-23) the ticket was rung up, store-local.",
  "order_channel": "How the order was placed: In-Store, Mobile App, Drive-Thru, Delivery.",
  "gross_sales": "quantity * unit_price before discounts, USD.",
  "discount_amount": "Promotional discount: LOYALTY10 = 10% off; HAPPYHOUR = 20% off beverages.",
  "cost_of_goods": "quantity * standard unit cost, USD.",
  "net_sales": "gross_sales - discount_amount, pre-tax."
 },
 "__SILVER__.fact_transactions": {
  "is_loyalty_transaction": "True if a Bean & Brew Rewards member was attached to the ticket.",
  "is_voided": "True if the ticket was voided at the register; excluded from sales metrics.",
  "net_sales": "Ticket net sales (gross - discounts), pre-tax, USD. Zero for voided tickets.",
  "sales_tax": "Sales tax collected based on store state rate."
 }
}

TAGS = [
 "ALTER SCHEMA __BRONZE__ SET TAGS ('layer' = 'bronze', 'domain' = 'sales')",
 "ALTER SCHEMA __SILVER__ SET TAGS ('layer' = 'silver', 'domain' = 'sales')",
 "ALTER SCHEMA __GOLD__ SET TAGS ('layer' = 'gold', 'domain' = 'sales')",
 "ALTER TABLE __SILVER__.fact_inventory_daily SET TAGS ('domain' = 'supply_chain')",
 "ALTER TABLE __GOLD__.inventory_current_status SET TAGS ('domain' = 'supply_chain')",
 "ALTER TABLE __SILVER__.fact_labor_daily SET TAGS ('domain' = 'hr')",
 "ALTER TABLE __SILVER__.dim_loyalty_member ALTER COLUMN email SET TAGS ('contains_pii' = 'true')",
 "ALTER TABLE __BRONZE__.raw_loyalty_members ALTER COLUMN email SET TAGS ('contains_pii' = 'true')",
 "ALTER TABLE __GOLD__.store_daily_performance SET TAGS ('owner_team' = 'analytics')",
 "ALTER TABLE __GOLD__.customer_360 SET TAGS ('owner_team' = 'customer_data')"
]

# Point the definitions above at the bundle's schemas
CONSTRAINTS = [[fill(t, cfg), n, fill(ddl, cfg)] for t, n, ddl in CONSTRAINTS]
COLUMN_COMMENTS = {fill(t, cfg): cols for t, cols in COLUMN_COMMENTS.items()}
TAGS = [fill(s, cfg) for s in TAGS]

existing = {(f"{r.table_schema}.{r.table_name}", r.constraint_name) for r in spark.sql(fill(
    "SELECT table_schema, table_name, constraint_name FROM information_schema.table_constraints WHERE table_schema IN ('__BRONZE__', '__SILVER__', '__GOLD__')", cfg)).collect()}
added = 0
for table, name, ddl in CONSTRAINTS:  # primary keys first, then the foreign keys that reference them
    if (table, name) in existing:
        continue
    added += 1
    if ddl.startswith("PRIMARY KEY"):  # key columns must be NOT NULL; CTAS doesn't always infer that
        for col in ddl[ddl.index("(") + 1:ddl.index(")")].split(","):
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {col.strip()} SET NOT NULL")
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} {ddl}")
print(f"Added {added} primary and foreign keys ({len(CONSTRAINTS) - added} already existed)")

n = 0
for table, cols in COLUMN_COMMENTS.items():
    for col, text in cols.items():
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN `{col}` COMMENT '{text.replace(chr(39), chr(92) + chr(39))}'")
        n += 1
print(f"Set {n} column comments")

# Tags may be governed by account-level tag policies with restricted values, so a failed tag is a warning
failed = []
for stmt in TAGS:
    try:
        spark.sql(stmt)
    except Exception as e:
        failed.append((stmt, str(e).splitlines()[0][:160]))
print(f"Applied {len(TAGS) - len(failed)} of {len(TAGS)} tags")
for stmt, err in failed:
    print(f"  WARNING tag not applied: {stmt}\n    {err}")